Connect To Drive


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/Projects/multi-agent-discovery'
print("Project root:", PROJECT_ROOT)
print("Exists:", os.path.exists(PROJECT_ROOT))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/Projects/multi-agent-discovery
Exists: True


Unzip MovieLens

In [ ]:
import zipfile

zip_path   = os.path.join(PROJECT_ROOT, 'data/raw/ml-32m.zip')
extract_to = os.path.join(PROJECT_ROOT, 'data/raw')

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_to)

extracted_dir = os.path.join(extract_to, 'ml-32m')
print("Extracted to:", extracted_dir)
print("Files:", sorted(os.listdir(extracted_dir)))

Extracted to: /content/drive/MyDrive/Projects/multi-agent-discovery/data/raw/ml-32m
Files: ['README.txt', 'checksums.txt', 'links.csv', 'movies.csv', 'ratings.csv', 'tags.csv']


Load the small core files, and peek at the big one

In [ ]:
DATA = os.path.join(PROJECT_ROOT, 'data/raw/ml-32m')

movies = pd.read_csv(os.path.join(DATA, 'movies.csv'))   # movieId, title, genres
links  = pd.read_csv(os.path.join(DATA, 'links.csv'))    # movieId, imdbId, tmdbId  <- bridge to TMDB

print("movies:", movies.shape)
print(movies.head(3), "\n")
print("links:", links.shape)
print(links.head(3), "\n")

ratings_path = os.path.join(DATA, 'ratings.csv')
print(f"ratings.csv size: {os.path.getsize(ratings_path)/1e6:.1f} MB")
print(pd.read_csv(ratings_path, nrows=5))   # peek only — do NOT load all 32M rows today

movies: (87585, 3)
   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance   

links: (87585, 3)
   movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0 

ratings.csv size: 877.1 MB
   userId  movieId  rating  timestamp
0       1       17     4.0  944249077
1       1       25     1.0  944250228
2       1       29     2.0  943230976
3       1       30     5.0  944249077
4       1       32     5.0  943228858


Smoke-test the TMDB key

In [ ]:
from google.colab import userdata
import requests

TMDB_API_KEY = userdata.get('TMDB_API_KEY')   # reads from Colab Secrets — the key never appears in the notebook

# Fetch one known movie: TMDB id 550 = "Fight Club"
url = "https://api.themoviedb.org/3/movie/550"
resp = requests.get(url, params={"api_key": TMDB_API_KEY}, timeout=10)

print("HTTP status:", resp.status_code)        # 200 = success ; 401 = bad/missing key
data = resp.json()
print("Title   :", data.get("title"))
print("Overview:", (data.get("overview") or "")[:120], "...")

HTTP status: 200
Title   : Fight Club
Overview: A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of th ...
